In [61]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString
import os


In [62]:
import pickle
data_path = "../../data/mydata/"
with open(os.path.join(data_path,"network_porto/porto_edges_new_simplify.pkl"), 'rb') as f:
    edgeinfo = pickle.load(f)
with open(os.path.join(data_path,"network_porto/porto_nodes_new.pkl"), 'rb') as f:
    nodeinfo = pickle.load(f)

In [63]:
print(edgeinfo[0])

['motorway_link', 32.3884588871153, '25503936', '4722746638']


In [64]:
print(nodeinfo[edgeinfo[0][-1]])

(-8.6409114, 41.1662762, 3.0)


In [65]:
geometry = [Point(lon, lat) for lon, lat, _ in nodeinfo.values()]
node_gdf = gpd.GeoDataFrame(list(nodeinfo), geometry=geometry,)

In [66]:
import osmnx as ox
import networkx as nx

In [67]:
graph = ox.graph_from_place("Porto, Portugal", network_type='drive')
print(f"Network loaded! Total live edges in Porto: {len(graph.edges)}")

Network loaded! Total live edges in Porto: 10974


In [68]:
print(len(edgeinfo))
print(len(nodeinfo))

10614
5062


In [69]:
min_lat, min_lon, max_lat, max_lon = float('inf'), float('inf'), float('-inf'), float('-inf')

for _, (lon, lat, _) in nodeinfo.items():
    min_lat = min(min_lat, lat)
    max_lat = max(max_lat, lat)
    min_lon = min(min_lon, lon)
    max_lon = max(max_lon, lon)

print(f"Bounding box: ({min_lat}, {min_lon}), ({max_lat}, {max_lon})")

Bounding box: (41.1406333, -8.6886988), (41.1858412, -8.5559061)


In [ ]:
bbox = (min_lon, min_lat, max_lon, max_lat)

graph = os.graph_from

In [71]:
nodes, edges = ox.graph_to_gdfs(graph) 
edges_metric = edges.to_crs(epsg=3763)

print(f"Historical network loaded! Found {len(edges)} road segments.")

Historical network loaded! Found 14214 road segments.


In [72]:
print(len(nodes))
print(len(edges))

6790
14214


In [73]:
print(type(edgeinfo))
sample_nodeinfo = iter(nodeinfo.items()).__next__()
sample_edgeinfo = iter(edgeinfo.items()).__next__()
print(sample_nodeinfo)
print(sample_edgeinfo)

<class 'dict'>
('25503936', (-8.6406364, 41.1660713, 3.0))
(0, ['motorway_link', 32.3884588871153, '25503936', '4722746638'])


In [74]:
valid_edges = []
invalid_edges = []

for i, edge in edgeinfo.items():
    # Extract u and v, ensuring they are integers (OSMnx expects int IDs)
    u = int(edge[2])
    v = int(edge[3])
    
    # Check if this exact directional edge exists in the live Porto graph
    if graph.has_edge(u, v):
        valid_edges.append((i, edge))
    else:
        # Sometimes an edge exists in the opposite direction (v to u)
        # We can check for that to diagnose one-way mismatches
        if graph.has_edge(v, u):
            invalid_edges.append((i, edge,  "Exists, but opposite direction"))
        else:
            invalid_edges.append((i, edge, "Completely missing from graph"))

print("\n--- Validation Results ---")
print(f"Total Edges Checked: {len(edgeinfo)}")
print(f"Valid Edges: {len(valid_edges)}")
print(f"Invalid/Missing Edges: {len(invalid_edges)}")

# Print a few invalid ones if they exist to investigate
if invalid_edges:
    print("\nSample of invalid edges:")
    for inv_edge in invalid_edges[:5]:
        print(f"Edge {inv_edge[0]}: {inv_edge[1][2]} -> {inv_edge[1][3]}: {inv_edge[2]}")


--- Validation Results ---
Total Edges Checked: 10614
Valid Edges: 9214
Invalid/Missing Edges: 1400

Sample of invalid edges:
Edge 27: 25620743 -> 9764806876: Completely missing from graph
Edge 31: 25620759 -> 12241567151: Completely missing from graph
Edge 34: 25620960 -> 3554720424: Completely missing from graph
Edge 35: 25620960 -> 137873037: Completely missing from graph
Edge 36: 25620960 -> 25620947: Completely missing from graph


In [75]:
sample_invalid_edges = invalid_edges[0]
print(f"Sample invalid edge {sample_invalid_edges[0]}: {sample_invalid_edges[1][2]} -> {sample_invalid_edges[1][3]}, Reason: {sample_invalid_edges[2]}")

Sample invalid edge 27: 25620743 -> 9764806876, Reason: Completely missing from graph


In [76]:
for eid, edge, _ in invalid_edges:
    u, v = edge[2], edge[3]
    if nodeinfo.get(u) is None or nodeinfo.get(v) is None:
        print(f"Edge {eid} has missing node(s): {u if nodeinfo.get(u) is None else ''} {v if nodeinfo.get(v) is None else ''}")

In [78]:
print(nodeinfo[edgeinfo[12][2]])
print()

(-8.5825433, 41.1541532, 3.0)

